# Data Flywheel — Architecture Walkthrough

This notebook walks through the key components of the data-flywheel repo — what each service does, how they connect, and how data flows through the system.

It is designed to be read top-to-bottom as an interactive tour of the codebase.

---

## The Big Picture

A data flywheel is a self-reinforcing loop:

```
Production Traffic
      │
      ▼
[1] Curation      — filter · dedup · PII redaction
      │
      ▼
[2] Finetuning    — ICL (zero compute) + LoRA SFT (GPU)
      │
      ▼
[3] Evaluation    — LLM judge · latency · cost
      │
      ▼
[4] Promotion     — best candidate becomes production model
      │
      └──────────────────────────────────┐
                                         ▼
                               (next cycle begins)
```

Each stage is a Celery task. Results flow forward through a Celery chain.

---
## Component 1 — The Orchestrator (FastAPI)

**Location:** `orchestrator/main.py`, `orchestrator/api/`

The FastAPI app exposes the control plane:
- `POST /flywheel/run` — trigger a full cycle
- `POST /flywheel/resume/{run_id}` — resume a failed run
- `GET /flywheel/status/{run_id}` — poll run status
- `GET /flywheel/runs` — list recent runs

Each run is stored in MongoDB with a unique `run_id` and stage-by-stage status.

In [ ]:
import requests

BASE_URL = 'http://localhost:8000'

# Check the API is healthy
resp = requests.get(f'{BASE_URL}/health')
print('API status:', resp.json())

# List recent runs
resp = requests.get(f'{BASE_URL}/flywheel/runs?limit=5')
runs = resp.json()['runs']
print(f'\nRecent runs ({len(runs)}):')
for run in runs:
    print(f"  {run['run_id'][:8]}... status={run['status']} started={run['started_at']}")

In [ ]:
# Inspect the last run in detail
if runs:
    last_run_id = runs[0]['run_id']
    resp = requests.get(f'{BASE_URL}/flywheel/status/{last_run_id}')
    status = resp.json()
    print(f"Run: {last_run_id}")
    print(f"Status: {status['status']}")
    print(f"Stages:")
    for stage, details in status.get('stages', {}).items():
        print(f"  {stage}: {details.get('status', 'N/A')}")

---
## Component 2 — The Task Queue (Celery + Redis)

**Location:** `orchestrator/core/celery_app.py`, `orchestrator/core/flywheel_loop.py`, `orchestrator/workers/`

The flywheel pipeline runs as a Celery chain:

```python
chain(
    run_curation.s(run_id, config),
    run_finetuning.s(run_id, config),
    run_evaluation.s(run_id, config),
    run_promotion.s(run_id, config),
)
```

Each task receives the previous task's result automatically. If a stage fails, the run can be resumed from the last completed stage using `POST /flywheel/resume/{run_id}`.

Workers are `ForkPoolWorker` processes — 4 concurrent by default.

In [ ]:
import redis
import os
from dotenv import load_dotenv
load_dotenv()

r = redis.Redis.from_url(os.getenv('REDIS_URL', 'redis://localhost:6379/0'))
print('Redis ping:', r.ping())
print('Active Celery keys:', len(r.keys('celery*')))

# Show registered tasks
import sys
sys.path.insert(0, '..')
from orchestrator.core.celery_app import celery_app
print('\nRegistered tasks:')
for task in sorted(celery_app.tasks.keys()):
    if task.startswith('flywheel'):
        print(f'  {task}')

---
## Component 3 — The Curator

**Location:** `orchestrator/services/curator/`

Three sub-components:

| File | Responsibility |
|---|---|
| `pipeline.py` | Orchestrates the full curation flow |
| `filters.py` | Quality scoring, PII redaction, length filtering |
| `dedup.py` | MinHash LSH near-deduplication |

Curation is **heuristic-only** — no models, no GPU required. Fast and cheap.

Raw logs come from Elasticsearch. Curated datasets are saved to MongoDB.

In [ ]:
from orchestrator.services.curator.filters import FilterPipeline
from orchestrator.services.curator.dedup import MinHashDeduplicator

# Demonstrate the filter pipeline
sample_logs = [
    {'prompt': 'What is machine learning?', 'response': 'Machine learning is a subset of AI that enables computers to learn from data.', 'model': 'test', 'latency_ms': 100},
    {'prompt': 'Hi', 'response': 'Hello!', 'model': 'test', 'latency_ms': 50},  # too short — will be filtered
    {'prompt': 'What is ML?', 'response': 'Machine learning is a subset of AI that enables computers to learn from data.', 'model': 'test', 'latency_ms': 110},  # near-duplicate
]

fp = FilterPipeline(min_quality_score=0.7, max_token_length=2048, min_response_length=20, remove_pii=False)
filtered = fp.run(sample_logs)
print(f'Before filter: {len(sample_logs)} samples')
print(f'After filter:  {len(filtered)} samples')

dedup = MinHashDeduplicator(threshold=0.85)
deduped = dedup.run(filtered)
print(f'After dedup:   {len(deduped)} samples')

---
## Component 4 — The Customizer (LoRA SFT)

**Location:** `orchestrator/services/customizer/`

| File | Responsibility |
|---|---|
| `lora_sft.py` | Orchestrates ICL registration and LoRA SFT training |
| `hf_client.py` | HuggingFace Hub — upload datasets, push adapters |

Two experiment types:
- **ICL** — registers the base model for few-shot evaluation. No training.
- **LoRA SFT** — fine-tunes using TRL + PEFT directly in the worker. Detects CUDA automatically. On GPU runs 500 steps with 4-bit quantization. On CPU runs 10 steps as a smoke test.

Trained adapters are pushed to HuggingFace Hub as private repos.

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Show what the training config looks like
import yaml
with open('../configs/flywheel.yaml') as f:
    cfg = yaml.safe_load(f)
print('\nLoRA training config:')
for k, v in cfg.get('experiments', {}).items():
    print(f'  {k}: {v}')

In [ ]:
# Show candidate models
with open('../configs/models.yaml') as f:
    models_cfg = yaml.safe_load(f)

print(f"Teacher: {models_cfg['teacher']['model']}\n")
print('Candidates:')
for c in models_cfg['candidates']:
    print(f"  {c['id']}")
    print(f"    Groq inference: {c['model']}")
    print(f"    HF training:    {c['hf_model']}")
    print(f"    Experiments:    {c['experiments']}")
    print(f"    Target:         {c['target']}")

---
## Component 5 — The Evaluator

**Location:** `orchestrator/services/evaluator/`

| File | Responsibility |
|---|---|
| `benchmarks.py` | Orchestrates full evaluation — metrics + judge |
| `judge.py` | LLM-as-judge using Llama 3.3 70B via Groq |
| `metrics.py` | Latency and cost measurement via live inference |

For each experiment:
1. `MetricsCollector` runs inference on all eval samples, measuring latency and token costs
2. `LLMJudge` calls the teacher model for reference responses, then asks the judge to score 1-5
3. Results are logged to MLflow and stored in MongoDB

The judge prompt is position-unbiased and criterion-specific (accuracy, completeness, coherence).

In [ ]:
# Show the judge prompt
from orchestrator.services.evaluator.judge import _JUDGE_SYSTEM_PROMPT, _JUDGE_USER_TEMPLATE

print('=== JUDGE SYSTEM PROMPT ===')
print(_JUDGE_SYSTEM_PROMPT)
print('\n=== JUDGE USER TEMPLATE ===')
print(_JUDGE_USER_TEMPLATE)

In [ ]:
# Show promotion criteria
with open('../configs/eval_criteria.yaml') as f:
    criteria = yaml.safe_load(f)

print('Promotion criteria (ALL must pass):')
for k, v in criteria['promotion_criteria'].items():
    print(f'  {k}: {v}')

---
## Component 6 — The Model Registry (MongoDB)

**Location:** `orchestrator/services/deployment/`, MongoDB collections

MongoDB stores the full system state:

| Collection | Contents |
|---|---|
| `flywheel_runs` | Run status, stage results, config |
| `experiments` | Per-experiment status, metrics, adapter repo IDs |
| `datasets` | Curated samples with full text |
| `model_registry` | Promoted models with metrics and smoke test results |

Promotion is atomic — exactly one model is `production` at any time. The previous model is archived before the new one is set.

In [ ]:
from pymongo import MongoClient
import os

mongo = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017'))
db = mongo[os.getenv('MONGO_DB', 'data_flywheel')]

print('MongoDB collections:')
for col in db.list_collection_names():
    count = db[col].count_documents({})
    print(f'  {col}: {count} documents')

In [ ]:
# Show the latest promoted model
try:
    promoted = db.model_registry.find_one({'status': 'production'})
    if promoted:
        print('Current production model:')
        print(f"  model_id:    {promoted.get('model_id')}")
        print(f"  model_name:  {promoted.get('model_name')}")
        print(f"  accuracy:    {promoted.get('accuracy')}")
        print(f"  latency_p95: {promoted.get('latency_p95_ms')}ms")
        print(f"  promoted_at: {promoted.get('promoted_at')}")
    else:
        print('No production model yet — run make run-flywheel to complete a cycle.')
except Exception as e:
    print(f'Registry not available: {e}')

---
## Component 7 — Experiment Tracking (MLflow)

**Location:** `orchestrator/utils/mlflow.py`, `orchestrator/services/evaluator/benchmarks.py`

Every evaluation run is logged to MLflow:
- **Metrics:** accuracy, mean_score, latency_p50/p95/p99, cost_per_1k, total_cost, sample_count
- **Tags:** experiment_id, model_id, model_name, run_id, evaluated_at
- **Promotion runs:** tagged separately with promotion metrics and smoke test results

View at: [http://localhost:5000](http://localhost:5000)

In [ ]:
import mlflow
import os

mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000'))

try:
    experiment = mlflow.get_experiment_by_name('data-flywheel')
    if experiment:
        print(f'Experiment: {experiment.name}')
        print(f'Experiment ID: {experiment.experiment_id}')
        
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        print(f'Total logged runs: {len(runs)}')
        if len(runs) > 0:
            print('\nRecent runs:')
            cols = [c for c in runs.columns if 'metrics.' in c or c in ['tags.model_name', 'tags.experiment_id']]
            print(runs[cols].head())
    else:
        print('No MLflow experiment found yet — run make run-flywheel first.')
except Exception as e:
    print(f'MLflow not available: {e}')

---
## Component 8 — The Scheduler (Celery Beat)

**Location:** `orchestrator/workers/scheduled.py`, `configs/flywheel.yaml`

Celery Beat runs the flywheel automatically on a cron schedule. The default is every 6 hours.

Before triggering a cycle it checks if there are enough new logs (`min_new_logs: 500`). If not, it skips and waits for the next tick.

```yaml
schedule:
  cron: "0 */6 * * *"   # every 6 hours
  min_new_logs: 500      # skip if fewer new logs
```

This means the flywheel only trains when there is enough new production data — preventing wasted compute on tiny datasets.

In [ ]:
# Show the schedule config
with open('../configs/flywheel.yaml') as f:
    cfg = yaml.safe_load(f)

print('Flywheel schedule:')
print(f"  cron:         {cfg['schedule']['cron']}")
print(f"  min_new_logs: {cfg['schedule']['min_new_logs']}")
print('\nCuration config:')
for k, v in cfg.get('curation', {}).items():
    print(f'  {k}: {v}')

---
## Summary

| Component | Technology | Location |
|---|---|---|
| API | FastAPI | `orchestrator/api/` |
| Task queue | Celery + Redis | `orchestrator/workers/` |
| Curation | Presidio + MinHash | `orchestrator/services/curator/` |
| Fine-tuning | TRL + PEFT + HF Hub | `orchestrator/services/customizer/` |
| Evaluation | Groq LLM judge | `orchestrator/services/evaluator/` |
| Model registry | MongoDB | `orchestrator/services/deployment/` |
| Experiment tracking | MLflow + PostgreSQL | `orchestrator/utils/` |
| Scheduler | Celery Beat | `orchestrator/workers/scheduled.py` |
| Log store | Elasticsearch | Docker service |
| State store | MongoDB | Docker service |

The full pipeline runs end-to-end in ~40-50 minutes on an NVIDIA H100 80GB.